# Challenge 05 - Digit Recognition (MLP)

Recognition of the digit (0-9) pronounced in each audio track with an MLP (`Dense` and `Dropout` layers only) trained with TensorFlow / Keras. The input is a fixed-size representation of the MFCCs that keeps the temporal structure: the speech part of each track is cropped using the energy (c0), the cepstral mean is removed, and the segment is resampled to a fixed number of time steps. The final classifier is an ensemble of 5 MLPs that average their output probabilities. Each test track is classified independently of the others. The predictions are saved in `Es7Predictions.txt`.

## Setup

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import warnings
warnings.filterwarnings('ignore', message='urllib3 v2 only supports OpenSSL')

import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Configuration
db_path = '../../DBs/AudioDigits_ML26_27'
train_file = 'audio_ml2627_train.pkl'
test_file = 'audio_ml2627_test_no_labels.pkl'
result_path = '../Es7Predictions.txt'

# Temporal features
n_steps = 16            # time steps of the resampled speech segment
smooth_width = 5        # moving average (frames) applied to c0 before the speech detection
min_speech_frames = 8   # shorter detections are discarded (the whole track is used)

# MLP
hidden_units = (256, 128)
dropout_in = 0.2
dropout_hidden = 0.4
l2_reg = 1e-3
learning_rate = 1e-3
n_epochs = 100
batch_size = 32

seed = 1234
n_models_val = 2        # networks per configuration in the comparison
n_models = 5            # networks per configuration in the final ensemble
top_k = 4               # best feature configurations combined in the final ensemble

# Feature configurations compared in the evaluation
CONFIGS = {
    'global only':        dict(temporal=False, global_feat=True),
    'span 0.55':          dict(temporal=True, global_feat=False, mode='span', speech_frac=0.55, cmn=True),
    'span 0.65':          dict(temporal=True, global_feat=False, mode='span', speech_frac=0.65, cmn=True),
    'span 0.75':          dict(temporal=True, global_feat=False, mode='span', speech_frac=0.75, cmn=True),
    'span 0.85':          dict(temporal=True, global_feat=False, mode='span', speech_frac=0.85, cmn=True),
    'peak 0.40':          dict(temporal=True, global_feat=False, mode='peak', speech_frac=0.40, cmn=True),
    'peak 0.55':          dict(temporal=True, global_feat=False, mode='peak', speech_frac=0.55, cmn=True),
    'peak 0.70':          dict(temporal=True, global_feat=False, mode='peak', speech_frac=0.70, cmn=True),
    'span 0.65 no CMN':   dict(temporal=True, global_feat=False, mode='span', speech_frac=0.65, cmn=False),
    'span 0.65 + global': dict(temporal=True, global_feat=True,  mode='span', speech_frac=0.65, cmn=True),
}

## Helper Functions

Dataset loading, extraction of the fixed-size features, model definition, training of the ensemble, prediction and export.

Features of a track (all computed from the track alone):

- **Speech detection**: c0 is smoothed and compared with the relative threshold `min(c0) + frac * (max(c0) - min(c0))`, which adapts to the noise level of each recording. In `span` mode the track is cropped between the first and the last frame above the threshold; in `peak` mode only the contiguous region around the energy peak is kept (more robust to noise bursts, but it can cut a word with an internal energy dip). The best mode and `frac` are chosen with the session validation.
- **Cepstral mean normalization (CMN)**: the mean of c1-c19 over the segment is subtracted, to reduce the effect of the channel and of the stationary noise.
- **Relative energy**: c0 minus its maximum, instead of the absolute c0.
- **Resampling**: the segment (20 coefficients x T frames) is linearly interpolated to `n_steps` frames and flattened, so tracks of different length give vectors of the same size. The log of the segment length is appended as an extra feature.

The scaler is fitted on the training patterns only.

In [2]:
def load_dataset(db_path, filename):
    return np.load(os.path.join(db_path, filename), allow_pickle=True)


def speech_segment(m, frac, mode):
    pad = smooth_width // 2
    c0 = np.convolve(np.pad(m[:, 0], pad, mode='edge'), np.ones(smooth_width) / smooth_width, mode='valid')
    thr = c0.min() + frac * (c0.max() - c0.min())
    if mode == 'span':
        idx = np.where(c0 >= thr)[0]
        lo, hi = idx[0], idx[-1]
    else:
        lo = hi = c0.argmax()
        while lo > 0 and c0[lo - 1] >= thr:
            lo -= 1
        while hi < len(c0) - 1 and c0[hi + 1] >= thr:
            hi += 1
    if hi - lo + 1 < min_speech_frames:
        return m
    return m[lo:hi + 1]


def resample(m, n):
    old, new = np.linspace(0, 1, len(m)), np.linspace(0, 1, n)
    return np.stack([np.interp(new, old, m[:, j]) for j in range(m.shape[1])], axis=1)


def temporal_features(m, frac, cmn, mode):
    m = np.asarray(m, dtype=float)
    seg = speech_segment(m, frac, mode)
    energy = seg[:, :1] - seg[:, 0].max()
    coef = seg[:, 1:]
    if cmn:
        coef = coef - coef.mean(axis=0)
    x = resample(np.hstack([energy, coef]), n_steps).ravel()
    return np.append(x, np.log(len(seg)))


def build_features(data, cfg):
    parts = []
    if cfg['temporal']:
        parts.append(np.array([temporal_features(m, cfg['speech_frac'], cfg['cmn'], cfg['mode']) for m in data['mfccs']]))
    if cfg['global_feat']:
        parts.append(np.array(data['features'], dtype=float))
    return np.hstack(parts)


def build_model(n_features):
    reg = tf.keras.regularizers.l2(l2_reg)
    layers = [tf.keras.layers.Input(shape=(n_features,)), tf.keras.layers.Dropout(dropout_in)]
    for units in hidden_units:
        layers.append(tf.keras.layers.Dense(units, activation='relu', kernel_initializer='he_normal', kernel_regularizer=reg))
        layers.append(tf.keras.layers.Dropout(dropout_hidden))
    layers.append(tf.keras.layers.Dense(10, activation='softmax'))
    model = tf.keras.Sequential(layers)
    model.compile(loss='sparse_categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(learning_rate), metrics=['accuracy'])
    return model


def fit_predict(x_tr, y_tr, x_te, n, seed0):
    scaler = StandardScaler().fit(x_tr)
    x_tr, x_te = scaler.transform(x_tr), scaler.transform(x_te)
    proba = 0
    for i in range(n):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed0 + i)
        model = build_model(x_tr.shape[1])
        model.fit(x_tr, y_tr, epochs=n_epochs, batch_size=batch_size, shuffle=True, verbose=0)
        proba = proba + model.predict(x_te, verbose=0)
    return proba / n


def session_proba(x, y, sess, n):
    out = []
    for a, c in [(1, 4), (4, 1)]:
        tr, va = sess == a, sess == c
        out.append((fit_predict(x[tr], y[tr], x[va], n, seed), y[va]))
    return out


def save_predictions(predictions, path):
    np.savetxt(path, np.asarray(predictions, dtype=int), fmt='%i')
    print(f'Predictions successfully saved to {path}')

## Data Loading

The training set contains sessions 1 and 4 (620 tracks), the test set sessions 2 and 3 (620 tracks, without labels). The label is the pronounced digit (`digits`). The speaker (`person_ids`) is not used. The `mfccs` are used for the temporal features and `features` for the global ones.

In [3]:
train_data = load_dataset(db_path, train_file)
test_data = load_dataset(db_path, test_file)

train_y = np.array(train_data['digits'], dtype=np.int32)
train_sess = np.array(train_data['session_ids'])

print('Training tracks: %d, test tracks: %d' % (len(train_y), len(test_data['mfccs'])))
print('Tracks per digit:', np.bincount(train_y))
print('Frames per track: %d-%d' % (min(len(m) for m in train_data['mfccs']), max(len(m) for m in train_data['mfccs'])))

Training tracks: 620, test tracks: 620
Tracks per digit: [62 62 62 62 62 62 62 62 62 62]
Frames per track: 79-745


## Model: Ensemble of MLPs

Each network is `Dropout(0.2)` on the input, two `Dense` layers (256 and 128 units, ReLU, He initialization, L2 regularization) each followed by `Dropout(0.4)`, and a softmax output with 10 units. They are trained with Adam for a fixed number of epochs (there is no early stopping because the validation data are other sessions, not a random hold-out). The final prediction is the argmax of the average of the output probabilities of an ensemble: 5 networks with different seeds for each of the `top_k` best feature configurations (different speech detection thresholds and modes). Combining configurations makes the result less dependent on the exact threshold of the speech detection, which is the most sensitive step.

In [4]:
n_features = build_features(train_data, CONFIGS['span 0.55']).shape[1]
build_model(n_features).summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dropout (Dropout)               │ (None, 321)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 116,618 (455.54 KB)

 Trainable params: 116,618 (455.54 KB)

 Non-trainable params: 0 (0.00 B)

## Model Evaluation

The split is done by session to simulate new recording conditions: train on one session and validate on the other, in both directions. Each feature configuration is compared with the same network and the same number of epochs. The `top_k` configurations with the highest mean accuracy are then combined in the full ensemble, which is evaluated with the same split. Since the configurations are selected on the same validation data, this ensemble accuracy is slightly optimistic.

In [5]:
results = {}
features = {}
for name, cfg in CONFIGS.items():
    features[name] = build_features(train_data, cfg)
    cv = session_proba(features[name], train_y, train_sess, n_models_val)
    accs = [accuracy_score(y, p.argmax(axis=1)) for p, y in cv]
    results[name] = np.mean(accs)
    print('%-32s 1->4: %.4f   4->1: %.4f   mean: %.4f' % (name, accs[0], accs[1], results[name]))

selected = sorted(results, key=results.get, reverse=True)[:top_k]
print('\nSelected configurations:', selected)

global only                      1->4: 0.3129   4->1: 0.3613   mean: 0.3371


span 0.55                        1->4: 0.4774   4->1: 0.7903   mean: 0.6339


span 0.65                        1->4: 0.6613   4->1: 0.8806   mean: 0.7710


span 0.75                        1->4: 0.7613   4->1: 0.7968   mean: 0.7790


span 0.85                        1->4: 0.7290   4->1: 0.7387   mean: 0.7339


peak 0.40                        1->4: 0.5871   4->1: 0.7839   mean: 0.6855


peak 0.55                        1->4: 0.6484   4->1: 0.7871   mean: 0.7177


peak 0.70                        1->4: 0.7419   4->1: 0.7484   mean: 0.7452


span 0.65 no CMN                 1->4: 0.6710   4->1: 0.8742   mean: 0.7726


span 0.65 + global               1->4: 0.6710   4->1: 0.8871   mean: 0.7790

Selected configurations: ['span 0.75', 'span 0.65 + global', 'span 0.65 no CMN', 'span 0.65']


In [6]:
cv = [session_proba(features[name], train_y, train_sess, n_models) for name in selected]
print('Ensemble of %d configurations x %d MLPs' % (len(selected), n_models))
for d, (a, c) in enumerate([(1, 4), (4, 1)]):
    proba = sum(run[d][0] for run in cv) / len(cv)
    acc = accuracy_score(cv[0][d][1], proba.argmax(axis=1))
    print('Train session %d -> validation session %d: accuracy %.4f' % (a, c, acc))

Ensemble of 4 configurations x 5 MLPs
Train session 1 -> validation session 4: accuracy 0.7516
Train session 4 -> validation session 1: accuracy 0.9032


## Prediction and Export

The ensemble is retrained on all training tracks (sessions 1 and 4) for each selected configuration, and the same preprocessing (features and scaler fitted on the training set) is applied to the test set. Each test track is classified independently of the others. The predictions are saved in `Es7Predictions.txt`, one digit per line.

In [7]:
proba = 0
for name in selected:
    x_train = build_features(train_data, CONFIGS[name])
    x_test = build_features(test_data, CONFIGS[name])
    proba = proba + fit_predict(x_train, train_y, x_test, n_models, seed)
predictions = proba.argmax(axis=1)

assert len(predictions) == 620
print('Predicted tracks per digit:', np.bincount(predictions, minlength=10))
save_predictions(predictions, result_path)

Predicted tracks per digit: [63 64 56 67 49 60 60 57 65 79]
Predictions successfully saved to ../Es7Predictions.txt
